# Python `requests` Library — Detailed, Practical Tutorial (Single Markdown Cell)

This tutorial explains **what each part of the `requests` library does, why it exists,
and when you should use it**, using clear explanations and real-world intuition.
It is written for **data pipelines, APIs, and research workflows**.

All code examples are shown as **indented blocks**, not fenced blocks, so this remains
**one single Markdown cell**.

---

## 1. What is `requests`?

`requests` is a Python library that lets your code **talk to web servers** using HTTP.

Every time you:
- query PubMed
- fetch PMC XML
- call a REST API
- download a file

you are making an **HTTP request**.

---

## 2. Installing and importing

You only need to install it once per environment.

    pip install requests

Then import it in Python:

    import requests

---

## 3. HTTP methods — what they *mean*

HTTP methods describe **intent**, not just mechanics.

### GET — “Give me data”

Used when:
- you are retrieving information
- nothing on the server should change

    r = requests.get(url)

Examples:
- searching PubMed
- fetching article metadata
- downloading XML

---

### POST — “Here is data, do something with it”

Used when:
- you are sending data
- you want the server to process or store something

    r = requests.post(url, data=data)

Examples:
- submitting a form
- sending a JSON payload
- triggering a computation

---

### PUT / PATCH / DELETE (less common)

PUT:
- replaces an existing resource

PATCH:
- modifies part of a resource

DELETE:
- removes a resource

You’ll mostly see these in authenticated APIs.

---

## 4. Query parameters (`params`) — VERY IMPORTANT

Query parameters are **key–value pairs appended to the URL**.

Instead of manually constructing URLs, always do this:

    params = {
        "q": "liver transplant",
        "year": 2024
    }

    r = requests.get(url, params=params)

Why this matters:
- correct URL encoding
- safer
- easier to read
- avoids subtle bugs

---

## 5. Passing lists to APIs

APIs differ in how they accept lists. You must follow the API spec.

### Comma-separated lists (very common)

Used by NCBI, PubMed, PMC.

    ids = ["12345", "67890", "11121"]

    params = {
        "id": ",".join(ids)
    }

    r = requests.get(url, params=params)

---

### Repeated keys

Some APIs expect the same key multiple times.

    params = [
        ("id", "123"),
        ("id", "456"),
        ("id", "789"),
    ]

    r = requests.get(url, params=params)

Requests preserves the repeated keys correctly.

---

## 6. Request body — `data` vs `json`

This is one of the most common points of confusion.

### `data=` → form-encoded

    r = requests.post(url, data={"a": 1, "b": 2})

Used for:
- HTML forms
- legacy APIs

---

### `json=` → JSON payload (recommended)

    r = requests.post(url, json={"a": 1, "b": 2})

Why this is better:
- automatically converts to JSON
- sets `Content-Type: application/json`
- less error-prone

---

## 7. Headers — how you identify yourself

Headers describe:
- who you are
- what format you want
- how you are authenticated

    headers = {
        "Accept": "application/json",
        "User-Agent": "ResearchBot/1.0",
        "Authorization": "Bearer YOUR_API_KEY"
    }

    r = requests.get(url, headers=headers)

Many APIs **reject requests without proper headers**.

---

## 8. Understanding the response object

Every request returns a **Response object**.

Important attributes:

    r.status_code    # HTTP code (200, 404, 500)
    r.ok             # True if status < 400
    r.headers        # response headers
    r.url            # final URL after redirects

### `r.status_code`
**What it is:**  
The HTTP status code returned by the server. This is the **primary indicator** of whether your request succeeded or failed.

**Common values and meaning:**
- `200` → OK (request succeeded)
- `201` → Created (often returned after a successful POST)
- `301 / 302` → Redirect (resource moved)
- `400` → Bad request (invalid parameters or malformed request)
- `401 / 403` → Authentication or authorization error
- `404` → Resource not found
- `429` → Too many requests (rate limiting)
- `500+` → Server-side error

**Why it matters:**  
A request can reach the server successfully but still fail logically.  
`status_code` tells you *how the server evaluated your request*.

**Typical usage:**  
- Debugging API failures  
- Logging request outcomes  
- Deciding whether to retry, skip, or stop  

---

### `r.ok`
**What it is:**  
A convenience boolean provided by `requests`.

- `True` if `status_code < 400`
- `False` if `status_code ≥ 400`

**Why it exists:**  
It provides a quick success/failure check without inspecting numeric codes.

**When it’s useful:**  
- Quick experiments  
- Notebooks  
- Simple scripts

**Limitations:**  
- Does **not** explain *why* a request failed  
- Not sufficient for production pipelines  

---

### `r.headers`
**What it is:**  
A dictionary-like object containing **metadata about the response**, not the actual content.

**Common headers you’ll see:**
- `Content-Type` → Format of the response (`application/json`, `text/xml`, etc.)
- `Content-Length` → Size of the response body
- `Date` → When the response was generated
- `Retry-After` → How long to wait before retrying (rate-limited APIs)

**Why it matters:**  
Headers tell you **how to interpret the response body** and **how the server wants you to behave next**.

**Typical usage:**  
- Checking format before calling `.json()`  
- Handling rate limits responsibly  
- Debugging encoding or content issues  

---

### `r.url`
**What it is:**  
The **final URL** that was actually requested after:
- parameter encoding
- redirects
- normalization by `requests`

**Why it’s important:**  
- Confirms that query parameters were encoded correctly  
- Helps debug subtle issues in `params`  
- Useful for logging and reproducibility  

**Common scenario:**  
You pass a `params` dictionary, and the request doesn’t behave as expected.  
Inspecting `r.url` shows exactly what was sent to the server.

---

### How these pieces fit together

- `r.status_code` → *Did the server accept or reject my request?*  
- `r.ok` → *Quick yes/no success check*  
- `r.headers` → *What kind of data did I receive and under what rules?*  
- `r.url` → *What request did I actually make?*

---

### Best practice

In robust pipelines:
- Use `raise_for_status()` to enforce correctness  
- Use `status_code` for diagnostics and logging  
- Use `headers` to safely interpret the response  
- Use `url` to debug and reproduce requests  

Together, these attributes give you **full visibility into what happened during an HTTP request**.

---

## 9. Reading response content

Different formats require different access methods.

### Plain text (HTML, XML)

    text = r.text

### Raw bytes (PDFs, images, ZIP files)

    content = r.content

### JSON

    data = r.json()

⚠️ Only call `.json()` if the server actually returns JSON.

---

## 10. Status codes & error checking

HTTP status codes indicate success or failure.

- 2xx → success
- 4xx → client error
- 5xx → server error

### Best practice: ALWAYS do this

    r.raise_for_status()

Why:
- converts HTTP failures into Python exceptions
- prevents silent bugs
- essential for pipelines

---

## 11. Timeouts — never skip these

Without a timeout, your code can hang forever.

    r = requests.get(url, timeout=30)

This protects:
- scripts
- notebooks
- production jobs

---

## 12. Redirects

Requests follows redirects automatically.

    r.history   # list of redirect responses
    r.url       # final destination

You can disable redirects if needed.

---

## 13. Authentication patterns

### Username / password

    r = requests.get(url, auth=("user", "password"))

### Token / API key (most common)

    headers = {
        "Authorization": "Bearer TOKEN"
    }

    r = requests.get(url, headers=headers)

---

## 14. Sessions — CRITICAL for performance

Sessions reuse:
- TCP connections
- headers
- cookies

This is much faster for many requests.

    session = requests.Session()
    session.headers.update({"User-Agent": "MyBot/1.0"})

    r1 = session.get(url1)
    r2 = session.get(url2)

Use sessions in **loops and pipelines**.

---

## 15. Exceptions you should expect

Requests raises different exceptions:

    requests.Timeout
    requests.ConnectionError
    requests.HTTPError
    requests.RequestException

Example pattern:

    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
    except requests.Timeout:
        print("Timed out")
    except requests.HTTPError:
        print("HTTP error")
    except requests.RequestException as e:
        print("Request failed:", e)

---

## 16. Streaming large downloads

Used for large files (PMC XML, PDFs).

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        for chunk in r.iter_content(chunk_size=8192):
            process(chunk)

This avoids loading everything into memory.

---

## 17. Safe, reusable request function

    def safe_get(url, params=None, headers=None, timeout=30):
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        return r

This is a **production-grade pattern**.

---

## 18. Common mistakes to avoid

- Building URLs manually
- Forgetting timeouts
- Ignoring status codes
- Assuming JSON responses
- Not respecting rate limits

---

## 19. Mental model (important)

A request has:
- method
- URL
- headers
- parameters or body

A response has:
- status code
- headers
- content

Understanding this model makes all APIs easier.

---

## Big picture first (this will clear the confusion)

You are mixing **three different things** that always appear together, so they *feel* like one thing:

1. **HTTP** → the *delivery system*
2. **API** → the *rules / contract*
3. **JSON payload** → the *data being sent*

They are related, but **not the same**.

Think of it like this:

> **HTTP is the road**  
> **API is the traffic rules**  
> **JSON payload is the package in the truck**

---

## What is HTTP (again, very simply)?

**HTTP is just a communication protocol.**

It answers questions like:
- How do I send a request?
- How does the server reply?
- How do we label success vs failure?

HTTP defines:
- GET, POST, PUT, DELETE
- Status codes (200, 404, 500)
- Headers
- Request → Response structure

**HTTP does NOT care what the data means.**  
It only cares about *how* it is sent.

---

## What is an API (why it exists)?

An **API (Application Programming Interface)** is a **contract** that says:

> “If you send a request in *this* way,  
> I will give you data in *that* way.”

An API defines:
- Which URLs you can call
- Which HTTP method to use
- What inputs are allowed
- What outputs you’ll get

📌 **An API uses HTTP**, but HTTP alone is not an API.

---

## Where JSON fits in

### What is JSON?

**JSON (JavaScript Object Notation)** is just a **data format**.

It is:
- Structured
- Human-readable
- Easy for machines to parse

Example JSON:

    {
      "pmid": "123456",
      "title": "Liver Transplant Outcomes",
      "year": 2024
    }

JSON is **not HTTP**  
JSON is **not an API**

It’s just **a way to represent data**.

---

## What is a JSON payload?

A **payload** means:
> “The actual data being sent in a request or response”

So a **JSON payload** is:
> Data encoded in JSON format and sent inside an HTTP request or response

---

## When do you send a JSON payload?

Usually with **POST**, **PUT**, or **PATCH** requests.

Conceptually:

- HTTP says: “I am sending data”
- API says: “Here’s the structure I expect”
- JSON payload is: “Here is the actual data”

Example (conceptual, not code-heavy):

You send:
- Method: POST
- URL: `/submit`
- Payload: JSON describing the data

The server reads the JSON and processes it.

---

## Why APIs love JSON

JSON is popular because:
- Language-independent
- Lightweight
- Easy to validate
- Easy to store
- Easy to parse

That’s why most modern APIs say:
> “Send me JSON, and I’ll respond with JSON”

---

## Putting it all together (THIS is the key)

Let’s describe one request in words:

> “I send an **HTTP POST request**  
> to an **API endpoint**  
> with a **JSON payload**  
> and the server responds with JSON.”

Each part has a role:

| Component | Role |
|--------|------|
| HTTP | How the message is delivered |
| API | Rules for what is allowed |
| JSON payload | The actual data |

---

## Common misconception (very important)

People often say:
- “I’m calling the API”
- “I’m sending JSON”
- “I’m making an HTTP request”

They are talking about **different layers of the same action**.

---

## One-sentence clarity (use this to ground yourself)

- **HTTP** → communication protocol  
- **API** → agreement on how to use that protocol  
- **JSON payload** → the data sent using that protocol  

---

## Why this matters for you (practically)

When something breaks, you need to ask:
- Is the **HTTP request** malformed?
- Did I violate the **API contract**?
- Is my **JSON payload** wrong?

Understanding the separation is what turns confusion into confidence.

---

If you want next, I can:
- draw a **step-by-step request lifecycle**
- show **GET vs POST with JSON vs params**
- or explain this using **PubMed / PMC specifically**


## What are API keys?

An **API key** is a **unique identifier** used to tell an API **who is making a request**.

You can think of an API key as:
- a **digital ID**
- a **permission token**
- a way for the server to recognize and manage you as a user

APIs use keys to:
- track usage
- enforce rate limits
- prevent abuse
- (sometimes) grant access to restricted features

An API key is **not a password**, but it should still be treated as **private**.

---

## Why APIs require keys

Without API keys:
- anyone could spam the service
- no way to enforce fair usage
- no accountability

With API keys, the server can:
- allow higher request limits
- throttle abusive clients
- contact users if there are issues

---

## How API keys are usually sent

API keys are sent in one of three common ways:

### 1. As a query parameter (very common)
    ?api_key=YOUR_KEY

### 2. As a request header
    Authorization: Bearer YOUR_KEY
    or
    X-API-Key: YOUR_KEY

### 3. Via environment variables (best practice on your side)
The key is stored locally and **never hard-coded**.

---

## Example: NCBI API key (PubMed / PMC)

NCBI provides API keys to:
- increase rate limits
- reduce throttling
- support large-scale data access

### Without an API key
- ~3 requests per second

### With an API key
- up to ~10 requests per second

This matters a lot for pipelines.

---

## How to get an NCBI API key

1. Create an NCBI account
2. Log in to NCBI
3. Go to **Account Settings**
4. Generate an **API Key**
5. Copy it somewhere secure

---

## How to use an NCBI API key (recommended way)

### Step 1: Store it as an environment variable

    export NCBI_API_KEY="your_real_key_here"

(or in Python, temporarily)

    import os
    os.environ["NCBI_API_KEY"] = "your_real_key_here"

---

### Step 2: Attach it to requests

NCBI expects the key as a **query parameter**.

    import requests
    import os

    NCBI_API_KEY = os.getenv("NCBI_API_KEY")

    params = {
        "db": "pubmed",
        "term": "liver transplant",
        "retmax": 10,
        "api_key": NCBI_API_KEY
    }

    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params=params,
        timeout=30
    )

    r.raise_for_status()

---

## Why environment variables matter (IMPORTANT)

❌ Bad practice:
    api_key = "hardcoded-secret-key"

Why this is bad:
- keys leak into GitHub
- keys end up in logs
- keys get shared accidentally

✅ Good practice:
- store keys in environment variables
- read them at runtime

---

## What happens if your API key is missing or wrong

Common outcomes:
- slower rate limits
- HTTP 429 (Too Many Requests)
- HTTP 403 (Forbidden)
- requests silently throttled

That’s why you:
- always include the key
- always respect rate limits
- still add `time.sleep()` if required

---

## Key takeaway (important)

- API keys identify **who you are**, not **what you can do**
- HTTP delivers the request
- The API defines the rules
- The API key enforces fair usage

For NCBI specifically:
> API keys let you query PubMed and PMC faster and more reliably — essential for large-scale text mining.

---

If you want next, I can:
- explain **rate limiting vs API keys**
- show **what happens when you exceed limits**
- or help you design a **safe key-handling pattern** for your pipeline
## Final takeaway

If you understand:
- params
- json vs data
- headers
- raise_for_status
- sessions
- timeouts

You understand **almost everything that matters in `requests`**.

# Exercise: Setting and Testing API Keys (NCBI) — Multiple Ways

This exercise is designed to help you **set an API key in different ways**, **verify that it is actually being used**, and **observe the difference in behavior with and without the key**.

The goal is not just to “set a key”, but to **prove to yourself that it is active**.

All code examples are shown as **indented blocks** so this stays **one single Markdown cell**.

---

## Background (what you are testing)

NCBI behavior differs depending on whether an API key is present:

- **Without API key**
  - Lower rate limit (~3 requests/second)
- **With API key**
  - Higher rate limit (~10 requests/second)

You will:
1. Make requests **without** a key
2. Make requests **with** a key
3. Compare results to confirm the key is actually used

---

## Part A — Test NCBI WITHOUT an API key

### Exercise A1 — Explicitly remove API key from environment

Task:
1. In Python, delete the environment variable (if it exists)
2. Confirm it is gone

    import os

    os.environ.pop("NCBI_API_KEY", None)
    print("NCBI_API_KEY =", os.getenv("NCBI_API_KEY"))

Expected:
- Output should be `None`

In [5]:
import os 
os.environ.pop("NCBI_API_KEY",None) ## None, is for safety in case there is no NCBI_API_KEY
os.environ
print("NCBI_API_KEY=",os.getenv("NCBI_API_KEY"))


NCBI_API_KEY= None


### Exercise A2 — Make a request without an API key

Task:
1. Call `esearch.fcgi` **without** `api_key`
2. Print:
   - status code
   - response time
   - final URL

In [15]:
import time                 # Provides functions for measuring time (used here to time the request)
import requests             # HTTP library used to send requests to web APIs

# Dictionary of query parameters that will be sent to the NCBI ESearch API
params = {
    "db": "pubmed",          # Database to search (PubMed)
    "term": "liver transplant",  # Search query (keywords)
    "retmode": "json",       # Response format (JSON instead of XML)
    "retmax": 5,             # Maximum number of results to return
    "tool": "api_key_test",  # Name of your tool/script (recommended by NCBI), A short name identifying the script, application, or project making the request.
    "email": "your_email@example.com"  # Contact email (required by NCBI policy)
}

start = time.time()         # Record the current time before sending the request

# Send an HTTP GET request to the NCBI ESearch endpoint
r = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",  # API endpoint URL
    params=params,          # Query parameters appended to the URL
    timeout=30              # Maximum time (seconds) to wait before failing, Wait at most 30 seconds for the server to respond. If it takes longer than that, stop and raise an erro
)

elapsed = time.time() - start   # Compute how long the request took (in seconds)

print("Status:", r.status_code)  # HTTP status code (200 = success, 4xx/5xx = error)
print("Elapsed:", round(elapsed, 3), "seconds")  # Request duration (rounded)
print("URL:", r.url)             # Final URL with encoded query parameters

Status: 200
Elapsed: 0.181 seconds
URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=liver+transplant&retmode=json&retmax=5&tool=api_key_test&email=your_email%40example.com


In [32]:
import time 
import requests 

os.environ["NCBI_API_KEY"] = "YOUR_NCBI_API_KEY"  ##secret should not be shared on git
NCBI_API_KEY = os.environ["NCBI_API_KEY"]
NCBI_search_API= "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db" : "pubmed", 
    "term": "liver transplant",
    "retmode": "json",
    "retmax":5
}

start = time.time() 

r=requests.get(
    NCBI_search_API,
    params=params,
    timeout=30
)

elapsed = time.time() - start  

print(r.status_code)
print(r.url)
print(round(elapsed,3))

200
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=liver+transplant&retmode=json&retmax=5
0.189


Expected observations:
- Request succeeds
- No `api_key=` appears in the URL
- Response time is slightly slower

---

## Part B — Set API key via environment variable (BEST PRACTICE)

### Exercise B1 — Set API key in the environment

Task:
1. Set the API key using an environment variable
2. Verify it exists

In [29]:
import os

os.environ["NCBI_API_KEY"] = "PASTE_YOUR_REAL_KEY_HERE"
print("NCBI_API_KEY =", os.getenv("NCBI_API_KEY"))

NCBI_API_KEY = PASTE_YOUR_REAL_KEY_HERE


Expected:
- You should see your key printed (temporarily OK for learning)

---

### Exercise B2 — Use the API key automatically

Task:
1. Read the key from the environment
2. Add it to `params`
3. Repeat the same request as before

In [36]:
import time
import requests
import os

api_key = os.getenv("NCBI_API_KEY")

params = {
    "db": "pubmed",
    "term": "liver transplant",
    "retmode": "json",
    "retmax": 5,
    "api_key": NCBI_API_KEY
}

start = time.time()
r = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
    params=params,
    timeout=30
)
elapsed = time.time() - start

print("Status:", r.status_code)
print("Elapsed:", round(elapsed, 3), "seconds")
print("URL:", r.url)

Status: 200
Elapsed: 0.192 seconds
URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=liver+transplant&retmode=json&retmax=5&api_key=YOUR_NCBI_API_KEY


Expected observations:
- URL now contains `api_key=`
- Response time is slightly faster
- Behavior is otherwise identical

✅ This confirms your API key is being sent.

---

## Part C — Defensive check: warn if API key is missing

### Exercise C1 — Add a safety check

Task:
1. Write a check that warns you if the key is missing
2. Do not crash the program

In [37]:
if api_key is None:
    print("WARNING: No NCBI API key detected — using lower rate limit")
else:
    print("NCBI API key detected")

NCBI API key detected


Why this matters:
- Prevents accidental slow runs
- Makes pipelines self-aware

---

## Part D — Alternative method (hard-coded key — for learning only)

⚠️ This is **NOT recommended** for real projects.

### Exercise D1 — Pass API key directly (temporary test)

Task:
1. Define the key inline
2. Pass it via `params`

params["api_key"] = "PASTE_YOUR_REAL_KEY_HERE"

Expected:
- Request works
- URL shows the key

❌ Do not commit this to GitHub  
❌ Do not use in shared code

---

## Part E — Stress test to PROVE the key works

### Exercise E1 — Rapid-fire requests without API key

Task:
1. Remove `api_key` from params
2. Send 5 requests quickly in a loop
3. Observe behavior

In [49]:
params = {
    "db": "pubmed",
    "term": "liver transplant",
    "retmode": "json",
    "retmax": 100,
}

start=time.time()

for i in range(10):
    r = requests.get(NCBI_search_API, params=params, timeout=30)
    # print(i, r.status_code)

end=time.time()
elapsed=end-start

print(round(elapsed,3))

params = {
    "db": "pubmed",
    "term": "liver transplant",
    "retmode": "json",
    "retmax": 100,
    "api_key": NCBI_API_KEY
}

start=time.time()

for i in range(10):
    r = requests.get(NCBI_search_API, params=params, timeout=30)
    # print(i, r.status_code)

end=time.time()
elapsed=end-start

print(round(elapsed,3))

1.384
2.065


Expected:
- Works, but you may see delays if scaled up

---

### Exercise E2 — Rapid-fire requests WITH API key

Task:
1. Add `api_key` back
2. Repeat the same loop

Expected:
- More stable
- Faster
- Less risk of 429 errors

This confirms the key is **functionally active**.

---

## Part F — Best-practice pattern (final goal)

### Exercise F1 — Centralized API key handling

Task:
1. Write a helper that:
- reads API key from environment
- injects it automatically into params
- prints a warning if missing

Conceptual structure:

In [50]:
def add_ncbi_auth(params):
    api_key = os.getenv("NCBI_API_KEY")
    if api_key:
        params["api_key"] = api_key
    else:
        print("WARNING: No NCBI API key detected")
    return params

Use this helper everywhere.

---

## Final verification checklist

You are DONE when you can confidently say:

- I know **where my API key lives**
- I can prove **when it is being sent**
- I can run code **with or without** a key intentionally
- My pipeline warns me if the key is missing

---

## Reflection question (important)

Answer in one sentence:

> How can you tell, from the request alone, whether your API key is being used?

(Hint: look at `r.url` and rate behavior.)

---

If you want next, I can:
- turn this into an **auto-graded checklist**
- integrate this into your existing NCBI pipeline
- or add an exercise for **handling 429 rate-limit errors**

# Exercises: Mastering NCBI E-utilities + `requests` (lots of small coding tasks)

These exercises are designed to help you practice **every `requests` concept** you listed
(params, lists, headers, Response object, parsing, raise_for_status, timeouts, sessions,
exceptions, streaming) *using real NCBI endpoints*.

You can do them in order. Each one is small, and builds confidence fast.

**NCBI base URL** (E-utilities):
- `https://eutils.ncbi.nlm.nih.gov/entrez/eutils/`

Common endpoints:
- `esearch.fcgi`  → search for IDs
- `esummary.fcgi` → metadata summary for IDs
- `efetch.fcgi`   → fetch full records (XML/text) for IDs
- `elink.fcgi`    → link IDs between databases (PMID → PMCID, etc.)
- `einfo.fcgi`    → database information

**NCBI best practice parameters** (add these to every request):
- `tool`: a short tool name (e.g., `"hepaFM"`)
- `email`: your email (NCBI asks for this in polite use)
- `api_key`: your key if you have one

---

## Setup exercises (do once)

### Exercise 0A — Create a config + base params
**Goal:** avoid repeating yourself.

Task:
1. Define:
   - `BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"`
   - `TOOL = "your_tool_name"`
   - `EMAIL = "your_email"`
   - `API_KEY = os.getenv("NCBI_API_KEY")`
2. Make a `BASE_PARAMS` dict with `tool`, `email`, and include `api_key` only if it exists.

**Check:** print `BASE_PARAMS` and make sure it looks right.

---

In [55]:
## Define base parameters

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
TOOL = "hepa_fm"
EMAIL = "layaljbara4@gmail.com"
API_KEY = os.getenv("NCBI_API_KEY")

## Define base parameters dictionary
BASE_PARAMS = {
    "base": BASE,
    "tool": TOOL,
    "email": EMAIL,
    "api_key": API_KEY
}

print(BASE_PARAMS)

{'base': 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/', 'tool': 'hepa_fm', 'email': 'layaljbara4@gmail.com', 'api_key': 'YOUR_NCBI_API_KEY'}


### Exercise 0B — Write `safe_get(url, params)` helper
**Goal:** enforce good habits.

Requirements:
- uses `timeout=30`
- calls `raise_for_status()`
- returns the `Response`

**Bonus:** log `r.status_code` and `r.url` for debugging.

In [66]:
## Define parameters
params ={
    "db": "pubmed",
    "term": "liver transplant",
    "retmode": "json",
    "api_key" : BASE_PARAMS["api_key"]
    
}

## Make request

r=requests.get(
    BASE_PARAMS["base"] +"/esearch.fcgi",
    params=params,
    timeout=30
)
r.raise_for_status()


---

## Part A — Requests fundamentals using NCBI

### Exercise 1 — Inspect the Response object
Call `einfo.fcgi` for `db=pubmed`.

Tasks:
1. Print:
   - `r.status_code`
   - `r.ok`
   - `r.headers.get("Content-Type")`
   - `r.url`
2. Print the first 300 characters of `r.text`.

**Goal skill:** understand Response attributes.

---

### Exercise 2 — Use `params` correctly (no manual URL building)
Call `esearch.fcgi` with:
- `db=pubmed`
- `term="liver transplant"`
- `retmode="json"`
- `retmax=5`

Tasks:
1. Print `r.url` (confirm query encoding)
2. Parse JSON with `r.json()`
3. Extract and print the list of PMIDs (`idlist`)

**Goal skill:** params + JSON parsing.

---

### Exercise 3 — Trigger and handle a failure intentionally
Make a request with a wrong endpoint, e.g.:
- `.../notreal.fcgi`

Tasks:
1. Wrap the request in `try/except`
2. Catch `requests.RequestException` and print the error message

**Goal skill:** exceptions + debugging.

---

### Exercise 4 — Add a timeout on purpose and test it
Tasks:
1. Make a request with `timeout=0.001`
2. Catch `requests.Timeout`

**Goal skill:** timeout handling.

---

## Part B — PubMed search functionality (ESearch)

### Exercise 5 — Search with date filters
Search PubMed for:
- `liver transplant`
- date range 2020 to 2025 (use PubMed syntax)

Example term idea:
- `liver transplant AND ("2020/01/01"[PDAT] : "2025/12/31"[PDAT])`

Tasks:
1. Return `count`
2. Print: `Number of PubMed hits for X from 2020–2025 is: <count>`

**Goal skill:** structured querying + parsing JSON.

---

### Exercise 6 — Pagination with `retstart`
Goal: get 3 pages of PMIDs (e.g., 20 per page).

Tasks:
1. Set `retmax=20`
2. Loop `retstart` = 0, 20, 40
3. Combine into one list and print total unique IDs

**Goal skill:** batching/pagination.

---

### Exercise 7 — Use `usehistory=y` (advanced, very important)
Goal: use NCBI history server for big queries.

Tasks:
1. Call `esearch` with:
   - `usehistory="y"`
   - `retmode="json"`
2. Extract:
   - `webenv`
   - `querykey`
3. Print them (these allow later fetches without sending all IDs again)

**Goal skill:** scalable NCBI workflows.

---

## Part C — Get metadata (ESummary)

### Exercise 8 — ESummary for a list of PMIDs
Using 5 PMIDs from Exercise 2:

Tasks:
1. Call `esummary.fcgi` with:
   - `db=pubmed`
   - `id=",".join(pmids)`
   - `retmode="json"`
2. From the JSON, print for each PMID:
   - title
   - pubdate (or sortpubdate)
   - source (journal)

**Goal skill:** list params (comma-separated) + nested JSON.

---

### Exercise 9 — Validate content-type before parsing JSON
Tasks:
1. Before calling `.json()`, check:
   - `r.headers.get("Content-Type")`
2. If it does not contain `"json"`, raise a ValueError with a helpful message.

**Goal skill:** safe parsing using headers.

---

## Part D — Fetch records (EFetch) and XML parsing

### Exercise 10 — Fetch MEDLINE text for PMIDs
Tasks:
1. Use `efetch.fcgi` with:
   - `db=pubmed`
   - `rettype="medline"`
   - `retmode="text"`
2. Print the first 40 lines.

**Goal skill:** text responses (`r.text`).

---

### Exercise 11 — Fetch XML and parse a title
Tasks:
1. Use `efetch.fcgi` with:
   - `retmode="xml"`
2. Parse with `xml.etree.ElementTree`
3. Extract:
   - ArticleTitle
   - AbstractText (if exists)
4. Handle missing abstract gracefully.

**Goal skill:** XML parsing + robust checks.

---

## Part E — Linking PubMed ↔ PMC (ELink) (your main use case)

### Exercise 12 — Batch PMID → PMCID mapping
Goal: input a list of PMIDs and return a dict `{pmid: pmcid_or_None}`.

Tasks:
1. Call `elink.fcgi` with:
   - `dbfrom="pubmed"`
   - `db="pmc"`
   - `linkname="pubmed_pmc"`
   - `id=",".join(pmids)`
   - `retmode="xml"`
2. Parse XML:
   - For each `<LinkSet>`, find its PMID
   - Find linked PMC ids if present
3. If no PMCID exists, store `None`

**Goal skill:** batch requests + XML parsing + missing links.

---

### Exercise 13 — Confirm your mapping is correct
Tasks:
1. Pick 1 PMID that has a PMCID and 1 that doesn’t
2. Print the `<LinkSet>` subtree for each (or key tags)
3. Explain in 1 sentence why one has no PMCID

**Goal skill:** debugging XML structure.

---

### Exercise 14 — Rate limiting practice (polite API usage)
Tasks:
1. Add a loop that processes batches of 200 PMIDs
2. After each batch request, sleep:
   - 0.34 seconds if you have **no** API key
   - 0.12 seconds if you **do** have an API key

**Goal skill:** responsible usage + throughput planning.

---

## Part F — Headers, user-agent, and reproducibility

### Exercise 15 — Add headers + compare server behavior
Tasks:
1. Call any endpoint twice:
   - once with default headers
   - once with `User-Agent` and `Accept`
2. Print response headers for both and see if anything differs

**Goal skill:** headers + response inspection.

---

## Part G — Robust engineering patterns (sessions, retries, logging)

### Exercise 16 — Use a `Session` for many calls
Tasks:
1. Create `session = requests.Session()`
2. Put your `tool/email/api_key` in a helper that merges params
3. Use the session for:
   - 10 consecutive esummary calls
4. Compare runtime vs repeated `requests.get()` (roughly)

**Goal skill:** sessions + performance.

---

### Exercise 17 — Implement a retry wrapper for transient failures
Task: write `get_with_retries(url, params, retries=3)`.

Requirements:
- retries on:
  - `requests.Timeout`
  - `requests.ConnectionError`
  - HTTP 429 (rate limit) and 5xx
- uses exponential backoff:
  - wait 1s, 2s, 4s (or similar)

**Goal skill:** production robustness.

---

### Exercise 18 — Create a structured logger line
For every request, print a single log line like:

    [OK] 200 | endpoint=esearch | elapsed=0.34s | url=<...>

If it fails:

    [FAIL] 429 | endpoint=esearch | reason=rate_limit | url=<...>

**Goal skill:** observability.

---

## Part H — Mini projects (combine everything)

### Project 1 — “Search → Summarize → Link to PMCID”
Input: a PubMed query string.

Output:
1. top 20 PMIDs
2. for each PMID: title, year, journal
3. PMCID if available else None
4. Save results to JSON

Must use:
- params
- raise_for_status
- session
- timeouts
- batch calls (comma-separated IDs)
- elink parsing

---

### Project 2 — “PMC availability report”
Input: list of PMIDs (from your JSONL scanning).

Output:
- count of PMIDs
- count with PMCID
- count without PMCID
- save dict to `pmc_available.json` and `pmc_unavailable.json`

Must include:
- batching
- rate limiting
- robust exception handling with retries

---

## Extra challenge exercises (quick but powerful)

### Challenge A — Detect wrong `retmode`
Task:
1. Request `retmode="json"` but accidentally parse as XML
2. Catch the error and print:
   - Content-Type
   - first 200 chars
   - a friendly hint

### Challenge B — Defensive parsing
Task:
- Write `get_first_text(root, xpath, default=None)` helper for XML

### Challenge C — Validate API key usage
Task:
- Print a warning if `NCBI_API_KEY` is missing:
  “No API key detected; using slower rate limit.”

---

## Suggested order (if you want a plan)
1 → 2 → 5 → 8 → 12 → 14 → 16 → 17 → Projects

---

If you want, paste your current PMID→PMCID function and I’ll turn **Exercises 12–14**
into a guided “fill-in-the-blanks” worksheet using your exact code style.